## PyINE prompt result database demo

This notebook shows how to use the framework's prompt result database in order to store prompt results in a persistent fashion, and in order to load these results to avoid paying inference costs over and over again.

The code below uses a temporary (demo) database that will be located in your tmpdir, but note that by default, the framework will rely on a single, shared database (see `pyine.prompts.get_framework_db_path()` for its default location).

Note also that the cells in this notebook must be execution once and in order.

In [ ]:
import pyine.prompts
import pyine.utils.filesystem
import pyine.utils.llm_providers
import pyine.utils.reprod

pyine.utils.reprod.entrypoint_setup()  # loads dotenv variables, seeds, sets up logging, etc.

In [ ]:
# we will setup / initialize a new database in a temp location; will delete if it exists
# (note: you would usually want to keep this database around, but we erase to have a reproducible demo)
result_db_path = pyine.utils.filesystem.get_tmp_dir() / "demo.db"
if result_db_path.exists():
    result_db_path.unlink()

# note: to use the default persistent database, we would use `pyine.prompts.get_framework_db()` instead
result_db = pyine.prompts.PromptResultDB(result_db_path)
assert result_db.count_entries() == 0  # fresh new

In [ ]:
# we will do a few demos using the code_summary prompt below
prompt_config = pyine.prompts.PromptBuildConfig(prompt_name="code_summary")
openai_gpt4o = pyine.utils.llm_providers.get_model_from_provider(
    provider="openai",
    model="gpt-4o",
    temperature=1.0,
    max_tokens=1024,
)
prompt_chain_config = pyine.prompts.PromptChainBuildConfig(
    prompt=prompt_config,
    provider=openai_gpt4o,
)

example_snippet = """\
def calculate_area(length: float, width: float) -> float:
    area = length * width
    print(f"The area of the rectangle is: {area:.2f} square units")
    return area

length = float(input("Enter the length: "))
width = float(input("Enter the width: "))
calculate_area(length, width)
"""

In [ ]:
# instead of inserting/fetching prompting results into/from the database directly,
# we will use a utility function that does it for us while also store lots of useful metadata...
records = pyine.prompts.fetch_or_generate_prompt_results(
    identifier="example_snippet",
    input_variables={"code": example_snippet},
    prompt_chain_config=prompt_chain_config,
    db=result_db,
)
# since the database is empty, this should have invoked a code summary generation chain and created a new record
assert len(records) == 1
print(f"Prompt:\n{records[0].prompt}")
print("=================\n")
print(f"Result:\n{records[0].result}")
print(f"Creation time:\n{records[0].creation_meta.created_at.isoformat()}")

In [ ]:
# a second call using the same arguments should NOT invoke again; it should instead return the original results
records2 = pyine.prompts.fetch_or_generate_prompt_results(
    identifier="example_snippet",
    input_variables={"code": example_snippet},
    prompt_chain_config=prompt_chain_config,
    db=result_db,
)
assert len(records2) == 1
assert records2[0] == records[0]  # same record as the previous one

# however, a third call that requests at least two results will trigger one new invocation
records3 = pyine.prompts.fetch_or_generate_prompt_results(
    identifier="example_snippet",
    input_variables={"code": example_snippet},
    prompt_chain_config=prompt_chain_config,
    db=result_db,
    generate_until_result_count=2,
)
assert len(records3) == 2
assert records3[0] == records[0]  # first result should still be the original
assert records3[1] != records[0]  # but this one should be novel
print(f"New result:\n{records[0].result}")
print(f"Creation time:\n{records[0].creation_meta.created_at.isoformat()}")

In [ ]:
# to clean up elements from the database, you can use the following function:
result_db.delete_records(identifier="example_snippet")  # supports more args, e.g. prompt name
assert result_db.count_entries() == 0
print("cleanup complete!")